1. 특별한 의미를 나타내기 위해서 None을 반환하는 함수가 오류를 일으키기 쉬운 이유는 None 이나 다른 값(예를 들어 0이나 빈 문자열)이 조건식에서 False로 평가되기 때문이다.

2. 특별한 상황을 알리 때에 None을 반환하는 대신 예외를 일으키자. 문서화가 되어있다면 호출하는 코드에서 예외를 적절히 처리할 것을 기대할 수 있다.

In [8]:
def divide(a,b):
    try:
        return  a / b
    except ZeroDivisionError:
        return None

result = divide(0, 5)
if not result:
    print("Error")
    
## 개선 방안 1. 반환값을 2개로 나누어 튜플에 담는다.
def divide(a, b):
    try:
        return True, a / b
    except ZeroDivisionError:
        return False, None

success, result = divide(0, 5)
if not success:
    print("Error")
    
## none 반환을 그냥 하지말자.
def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError as e:
        raise ValueError("Cannot divide by zero") from e
    
## 호출부도 예외 처리
try:
    result = divide(0, 5)
    print(result)
except ValueError as e:
    print(f"Error: {e}")

Error
0.0


클로저가 변수 스코프와 상호작용하는 법
1. 클로저 함수는 자신의 정의된 스코프 중 어디에 있는 변수도 참조 가능하다.
2. 기본적으로 클로저에서 변수를 할당하면 바깥쪽 스코프에는 영향을 미치지 않는다.
3. 파이썬 3에서는 nonlocal 문을 사용하여 클로저를 감싸고 있는 스코프의 변수를 수정할 수 있음을 알린다.
4. 파이썬 2에서는 수정 가능한 값으로 nonlocal 문이 없는 문제를 우회한다.
5. 간단한 함수 이외에는 nonlocal문을 사용하지말자.

In [14]:
def sort_priority(values,group):
    def helper(x):
        if x in group:
            return (0, x)
        return (1, x)
    values.sort(key=helper)
    
numbers = [1, 2, 3, 4, 5, 6]
group = {2, 4, 6}
sort_priority(numbers, group)
print(numbers)  # [2, 4, 6, 1, 3, 5]

## 클로저란? 자신이 정의된 스코프에 있는 변수를 참조하는 함수이다.
## 좀더 명확해 보이는 방식으로 코드를 수정해보자

def sort_priority2(values, group):
    found = False
    def helper(x):
        if x in group:
            found = True
            return (0, x)
        return (1, x)
    values.sort(key=helper)
    return found

found = sort_priority2(numbers, group)
print('Found:', found)  
print(numbers)  # [2, 4, 6, 1, 3, 5]

## 왜 found가 False로 나오는가?
## python은 스코프를 결정할 때, 함수 안에서 변수가 할당되면 그 변수는 지역변수로 간주한다.

def sort_priority3(values, group):
    found = False
    def helper(x):
        nonlocal found
        if x in group:
            found = True
            return (0, x)
        return (1, x)
    values.sort(key=helper)
    return found

found = sort_priority3(numbers, group)
print('Found:', found)  
print(numbers)  # [2, 4, 6, 1, 3, 5]

## 하지만 nonlocal문은 함수가 단순한 경우에만 사용하자. 복잡한 경우에는 다른 방법을 고려하자.
## 대표적으로 클래스를 사용하는 방법이 있다.

class Sorter(object):
    def __init__(self, group):
        self.group = group
        self.found = False
        
    def __call__(self, x):
        if x in self.group:
            self.found = True
            return (0, x)
        return (1, x)
    
sorter = Sorter(group)
numbers.sort(key=sorter)
assert sorter.found is True

[2, 4, 6, 1, 3, 5]
Found: False
[2, 4, 6, 1, 3, 5]
Found: True
[2, 4, 6, 1, 3, 5]


리스트를 반환하는대신 제너레이터를 고려하자
1. 제너레이터를 사용하는 방법이 누적된 결과의 리스트를 반환하는 방법보다 이해하기에 명확하다.
2. 제너레이터에서 반환한 이터레이너는 제너레이터 함수의 본문에 있는 yield 표현식에 전달된 값들의 집합임.
3. 제너레이터는 모든 입출력을 메모리에 저장하지 않기 때문에 입력값의 양을 알기 어려울 때도 연속 출력을 만들 수 있다.

In [ ]:
def index_words(text):
    result = []
    if text:
        result.append(0)
    for index, letter in enumerate(text):
        if letter == ' ':
            result.append(index + 1)
    return result

## index_words 의 두가지 문제점이있다.
## 1. 코드가 약간 길다. 2. 모든 결과를 리스트에 저장해야한다. 만약 text가 매우 길다면, 메모리 사용량이 많아질 수 있다.
## 제너레이터를 사용하여 개선해보자.

def index_words_iter(txt):
    if text:
        yield 0
    for index, letter in enumerate(text):
        if letter == ' ':
            yield index + 1

## 제너레이터는 이터레이터에 상태가 있고 재사용할 수 없다. 따라서 제너레이터를 재사용하려면, 제너레이터를 반환하는 함수를 만들어야 한다.



인수를 순회할때는 방어적으로하자
1. 입력 인수를 여러번 순회하는 함수를 작성시 주의, 입력 인수가 이터레이터라면 값을 잃어버릴 수도있다.
2. 파이썬의 이터레이터 프로토콜은 컨테이너와 이터레이터가 내장 함수 iter, next와 for 루프 및 관련 표현식과 상호작용하는 방법을 정의한다.
3. __iter__ 메서드를 제너레이터로 구현하면 자신만의 이터러블 컨테이너 타입을 쉽게 정의가능하다.
4. 어떤 값에 iter를 두번 호출하였을 때 같은 결과가 나오고 내장 함수 next로 전진시킬 수 있다면 그 값은 컨테이너가 아닌 이터레이터다.

In [ ]:
def normalize(numbers):
    total = sum(numbers)
    result = []
    for value in numbers:
        percent = 100 * value / total
        result.append(percent)
    return result

## 이 함수는 방문리스트를 받아 동작한다.
visit_counts = [15, 35, 80]
percentages = normalize(visit_counts)
print(percentages)  # [10.0, 23.333333333333332, 66.66666666666666]

def read_visits(data_path):
    with open(data_path) as f:
        for line in f:
            yield int(line)
            
it = read_visits('my.txt')

print(list(it))
# it = read_visits('my.txt')
print(list(it))


### 이터레이터는 한 번만 순회할 수 있다. 따라서, read_visits를 여러번 호출해야한다.
# 이 문제를 해결하려면 입력 이터레이터를 명시적으로 소진하고 전체 콘텐츠의 복사본을 리스트에 저장해야한다.

def normalize_copy(numbers):
    numbers = list(numbers)
    total = sum(numbers)
    result = []
    for value in numbers:
        percent = 100 * value / total
        result.append(percent)
    return result
it = read_visits('my.txt')
percentages = normalize_copy(it)
print(percentages)

## 하지만 이 방식의 문제는 이터레이터 콘텐츠의 복사본이 클 수도 있다는 것이다.

def normalize_func(get_iter):
    total = sum(get_iter())
    result = []
    for value in get_iter():
        percent = 100 * value/total
        result.append(percent)
    return result

class ReadVisits(object):
    def __init__(self, data_path):
        self.data_path = data_path
        
    def __iter__(self):
        with open(self.data_path) as f:
            for line in f:
                yield int(line)
            
visits = ReadVisits('./my.txt')
percentage = normalize(visits)
print(percentage)

def normalize_defensive(numbers):
    if iter(numbers) is iter(numbers):
        raise TypeError('Must supply a container')
    total = sum(numbers)
    result = []
    for value in numbers:
        percent = 100 * value / total
        result.append(percent)
    return result

## 컨테이너는 오류 x
visits = [15,35,80]
normalize_defensive(visits)
## 이터레이터는 오류 반환 (방어로직)
it = iter(visits)
normalize_defensive(it)

[11.538461538461538, 26.923076923076923, 61.53846153846154]
[14, 35, 60]
[]
[12.844036697247706, 32.11009174311926, 55.04587155963303]
[12.844036697247706, 32.11009174311926, 55.04587155963303]


TypeError: Must supply a container

In [1]:
def log(message, values):
    if not values:
        print(message)
    else :
        values_str =', '.join(str(x) for x in values)
        print('%s: %s' % (message, values_str))

log('My numbers are',[1,2])
log('Hi there', [])

My numbers are: 1, 2
Hi there


In [2]:
def log(message, *values):
    if not values:
        print(message)
    else :
        values_str =', '.join(str(x) for x in values)
        print('%s: %s' % (message, values_str))

log('My numbers are',[1,2])
log('Hi there')

My numbers are: [1, 2]
Hi there


In [4]:
def my_generator():
    for i in range(10):
        yield i
def my_func(*args):
    print(args)

it = my_generator()
my_func(*it)

(0, 1, 2, 3, 4, 5, 6, 7, 8, 9)


In [5]:
def log(sequence, message, *values):
    if not values:
        print('%s: %s' % (sequence, message))
    else:
        values_str=','.join(str(x) for x in values)
        print('%s: %s: %s' % (sequence, message, values_str))
        
log(1, 'Favorites', 7, 33)
log('Favorite numbers', 7,33)

1: Favorites: 7,33
Favorite numbers: 7: 33


In [7]:
def remainder(number, divisor):
    return number % divisor

assert remainder(20,7) == 6

remainder(20,7)
remainder(divisor= 7 , number= 20)

6

In [11]:
from datetime import datetime
from time import sleep
def log(message, when=datetime.now()):
    print('%s: %s' % (when, message))
    
log('Hi there!')
sleep(0.1)
log('Hi again')



2026-07-12 22:13:03.109535: Hi there!
2026-07-12 22:13:03.109535: Hi again


In [ ]:
def log(message, when = None):
 
            
    when = datetime.now() if when is None else when
    print('%s: %s' % (when,message))
log('Hi there!')
sleep(0.1)
log('Hi again!')


2026-07-12 22:24:55.820387: Hi there!
2026-07-12 22:24:55.921347: Hi again!


In [15]:
import json
def decode(data, default={}):
    try:
        return json.loads(data)
    except ValueError:
        return default

In [17]:
foo = decode('bad data')
foo['stuff'] = 5
bar = decode('also bad')
bar['meep'] = 1
print('Foo:', foo)
print('Bar:', bar)


Foo: {'stuff': 5, 'meep': 1}
Bar: {'stuff': 5, 'meep': 1}


In [19]:
def decode(data,default = None):
    if default is None:
        default = {}
    try:
        return json.loads(data)
    except ValueError:
        return default
foo = decode('bad data')
foo['stuff'] = 5
bar = decode('also bad')
bar['meep'] = 1
print('Foo:', foo)
print('Bar:', bar)    

Foo: {'stuff': 5}
Bar: {'meep': 1}


In [22]:
def safe_division(number, divisor,*, ignore_overflow=True, ignore_zero_division=False):
    try:
        return number / divisor
    except OverflowError:
        if ignore_overflow:
            return 0
        else:
            raise 
    except ZeroDivisionError:
        if ignore_zero_division:
            return float('inf')
        else:
            raise
        
        
safe_A = safe_division(1, 10**500,True,False)

TypeError: safe_division() takes 2 positional arguments but 4 were given